# UCI Datathon Project 2026
## Data Loading and Initial Setup

In [2]:
import pandas as pd
import re
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


### Load Raw Data

In [3]:
# Read datasets from Drive
with open('/content/drive/My Drive/emoji.txt', 'r') as f:
    emoji_content = f.read()

with open('/content/drive/My Drive/tweets.txt', 'r') as f:
    tweets_content = f.read()

# Split and strip
tweets = [t.strip() for t in tweets_content.splitlines()]
emoji = [e.strip() for e in emoji_content.splitlines()]

### Create and Validate DataFrame

In [4]:
df = pd.DataFrame({
    "text": tweets,
    "label": emoji
})

print(f"Shape: {df.shape}")
display(df.head(10))

Shape: (225331, 2)


,text,label
0,RT @VibingOverHoes: Bet you'll get hungry htt...,heart_eyes
1,Starbucks employee confuses boyfriend by sayin...,yum
2,When your Starbucks store makes you an iced mo...,sob
3,"Being told ""girl your romper looks fierce!"" At...",blush
4,"I got a Starbucks drink at school today, shit ...",sob
5,So when is Starbucks getting their grill chees...,yum
6,really need @Jade_morgann to bring me some su...,weary
7,Bet you're feeling super salty now. https://t....,smirk
8,RT @Daniarmstrong88: So it's the end of #brang...,grin
9,@ooohkarluuuh and forgot to get me Starbucks,sob


### Exploratory Data Analysis
Check for missing values and class distribution.

In [5]:
print("Missing Values:")
print(df.isna().sum())

print("\nClass Distribution:")
print(df['label'].value_counts())

print(f"\nUnique Emojis: {df['label'].nunique()}")

Missing Values:
text     0
label    0
dtype: int64

Class Distribution:
label
sob           50525
heart_eyes    39193
weary         26855
blush         22894
wink          18078
yum           16790
smirk         15231
grin          15138
relaxed       10472
flushed       10155
Name: count, dtype: int64

Unique Emojis: 10


### Text Preprocessing
Defining the cleaning logic and applying it to the dataset.

In [6]:
def clean_tweet(text):
    text = text.lower()
    text = re.sub(r'^rt', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-zA-Z\s!?\']', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [7]:
df['cleaned_tweet'] = df['text'].apply(clean_tweet)
display(df[['text', 'cleaned_tweet']].head())

,text,cleaned_tweet
0,RT @VibingOverHoes: Bet you'll get hungry htt...,bet you'll get hungry
1,Starbucks employee confuses boyfriend by sayin...,starbucks employee confuses boyfriend by sayin...
2,When your Starbucks store makes you an iced mo...,when your starbucks store makes you an iced mo...
3,"Being told ""girl your romper looks fierce!"" At...",being told girl your romper looks fierce! at t...
4,"I got a Starbucks drink at school today, shit ...",i got a starbucks drink at school today shit t...


In [14]:
df['cleaned_tweet'] = df['text'].apply(clean_tweet)

# Remove duplicate tweets to prevent bias and overfitting
initial_count = len(df)
df = df.drop_duplicates(subset=['cleaned_tweet'])
final_count = len(df)

print(f"Removed {initial_count - final_count} duplicate tweets.")
print(f"New dataset shape: {df.shape}")

display(df[['text', 'cleaned_tweet']].head())

Removed 84729 duplicate tweets.
New dataset shape: (140602, 4)


,text,cleaned_tweet
0,RT @VibingOverHoes: Bet you'll get hungry htt...,bet you'll get hungry
1,Starbucks employee confuses boyfriend by sayin...,starbucks employee confuses boyfriend by sayin...
2,When your Starbucks store makes you an iced mo...,when your starbucks store makes you an iced mo...
3,"Being told ""girl your romper looks fierce!"" At...",being told girl your romper looks fierce! at t...
4,"I got a Starbucks drink at school today, shit ...",i got a starbucks drink at school today shit t...


In [8]:
pip install transformers torch accelerate

In [16]:
from transformers import TrainingArguments, Trainer

# Load pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label_map))

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch", # Corrected argument name from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
)

# Define metrics computation
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {
        'accuracy': accuracy,
        'f1_weighted': f1,
    }

# Create Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Starting BERT training...")
# Train the model
trainer.train()
print("BERT training complete.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

Starting BERT training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.734695,1.801585,0.359980,0.293340
2,1.677464,1.794807,0.363963,0.339431
3,1.195087,1.986192,0.354753,0.340517


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

BERT training complete.


In [15]:
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Load pre-trained BERT tokenizer
MODEL_NAME = 'bert-base-uncased'
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

# Map labels to integers for BERT
label_map = {label: i for i, label in enumerate(df['label'].unique())}
id_to_label = {i: label for label, i in label_map.items()}
df['encoded_label'] = df['label'].map(label_map)

class EmojiDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Split data into train and test sets
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['encoded_label'].values,
    random_state=42
)

# Tokenize the cleaned tweets for train and test sets separately
train_encodings = tokenizer(list(train_df['cleaned_tweet'].values),
                            truncation=True,
                            padding=True,
                            max_length=128)

test_encodings = tokenizer(list(test_df['cleaned_tweet'].values),
                           truncation=True,
                           padding=True,
                           max_length=128)

train_dataset = EmojiDataset(train_encodings, train_df['encoded_label'].values)
test_dataset = EmojiDataset(test_encodings, test_df['encoded_label'].values)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

/tmp/ipykernel_8211/39519831.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['encoded_label'] = df['label'].map(label_map)


Number of training samples: 112481
Number of test samples: 28121


# Finalized Project Structure
This notebook has been cleaned to maintain a modular workflow.

### Note on Cleanup
The previous redundant data loading and processing cells have been consolidated above for better readability.

In [ ]:
# Cell intentionally left blank as code was moved to modular section

In [ ]:
# Cell intentionally left blank as code was moved to modular section

### Final Preprocessing Output

In [ ]:
display(df.tail())

,text,label,cleaned_tweet
225326,RT @TheFunnyVine: She broke down as soon as sh...,grin,she broke down as soon as she heard starbucks
225327,@PinKC0ttNkandi_ Mines shorter than yours Lls ...,sob,mines shorter than yours lls it bet not start ...
225328,@BenColeyGolf was yours the 5k bet on US earli...,wink,was yours the k bet on us earlier today !?
225329,@Kevyn_Brown @BurgerKing come on leek i gotta ...,sob,come on leek i gotta at least once
225330,Just won £60 on a bet get in,smirk,just won on a bet get in


In [ ]:
print('Preprocessing Complete.')

Preprocessing Complete.


In [ ]:
import re

def clean_tweet(text):
    """
    Clean a raw tweet for emoji prediction.

    Design decisions:
    - Lowercase to normalize case variations
    - Remove URLs (no emotional signal)
    - Remove @mentions (usernames don't predict emoji)
    - Remove retweet prefix (e.g., 'rt ')
    - Keep hashtag words, remove # symbol (the word carries meaning)
    - Remove special characters/numbers (focus on language, i.e., keep only letters and spaces)
    - Collapse whitespace
    """
    text = text.lower()
    text = re.sub(r'^rt', '', text) # Remove retweet prefix
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-zA-Z\s!?\']', '', text) # Keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [ ]:
df['cleaned_tweet'] = df['text'].apply(clean_tweet)
print(df[['text', 'cleaned_tweet']].head())

                                                text  \
0  RT @VibingOverHoes: Bet you'll get hungry  htt...   
1  Starbucks employee confuses boyfriend by sayin...   
2  When your Starbucks store makes you an iced mo...   
3  Being told "girl your romper looks fierce!" At...   
4  I got a Starbucks drink at school today, shit ...   

                                       cleaned_tweet  
0                              bet you'll get hungry  
1  starbucks employee confuses boyfriend by sayin...  
2  when your starbucks store makes you an iced mo...  
3  being told girl your romper looks fierce! at t...  
4  i got a starbucks drink at school today shit t...  


In [ ]:
# Prepared for model training

TF-IDF SETUP

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score

# Features and labels
X = df["cleaned_tweet"]
y = df["label"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# TF-IDF settings
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_features=10000,
    sublinear_tf=True
)

Train size: 180264
Test size: 45067


In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("model", MultinomialNB())
])

print("Training Naive Bayes...")

nb_pipeline.fit(X_train, y_train)

nb_preds = nb_pipeline.predict(X_test)

nb_acc = accuracy_score(y_test, nb_preds)
nb_macro_f1 = f1_score(y_test, nb_preds, average="macro")
nb_weighted_f1 = f1_score(y_test, nb_preds, average="weighted")

print("Naive Bayes Results")
print("Accuracy:", nb_acc)
print("Macro F1:", nb_macro_f1)
print("Weighted F1:", nb_weighted_f1)

print("\nClassification Report:")
print(classification_report(y_test, nb_preds))

Training Naive Bayes...
Naive Bayes Results
Accuracy: 0.45547740031508643
Macro F1: 0.42942484751508125
Weighted F1: 0.44866749396419003

Classification Report:
              precision    recall  f1-score   support

       blush       0.35      0.34      0.35      4579
     flushed       0.89      0.41      0.56      2031
        grin       0.74      0.28      0.41      3028
  heart_eyes       0.54      0.55      0.55      7839
     relaxed       0.82      0.20      0.32      2094
       smirk       0.54      0.29      0.37      3046
         sob       0.41      0.66      0.51     10105
       weary       0.39      0.39      0.39      5371
        wink       0.35      0.43      0.39      3616
         yum       0.53      0.41      0.46      3358

    accuracy                           0.46     45067
   macro avg       0.56      0.39      0.43     45067
weighted avg       0.50      0.46      0.45     45067



In [ ]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

print("Training Logistic Regression...")

lr_pipeline.fit(X_train, y_train)

lr_preds = lr_pipeline.predict(X_test)

lr_acc = accuracy_score(y_test, lr_preds)
lr_macro_f1 = f1_score(y_test, lr_preds, average="macro")
lr_weighted_f1 = f1_score(y_test, lr_preds, average="weighted")

print("Logistic Regression Results")
print("Accuracy:", lr_acc)
print("Macro F1:", lr_macro_f1)
print("Weighted F1:", lr_weighted_f1)

print("\nClassification Report:")
print(classification_report(y_test, lr_preds))

Training Logistic Regression...
Logistic Regression Results
Accuracy: 0.4451372401091708
Macro F1: 0.42838797111829996
Weighted F1: 0.45083790778078336

Classification Report:
              precision    recall  f1-score   support

       blush       0.37      0.34      0.36      4579
     flushed       0.39      0.60      0.48      2031
        grin       0.39      0.38      0.39      3028
  heart_eyes       0.64      0.51      0.57      7839
     relaxed       0.28      0.40      0.33      2094
       smirk       0.34      0.46      0.39      3046
         sob       0.65      0.38      0.48     10105
       weary       0.40      0.48      0.44      5371
        wink       0.35      0.47      0.40      3616
         yum       0.41      0.52      0.46      3358

    accuracy                           0.45     45067
   macro avg       0.42      0.45      0.43     45067
weighted avg       0.48      0.45      0.45     45067



In [ ]:
from sklearn.svm import LinearSVC

svm_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("model", LinearSVC(
        class_weight="balanced"
    ))
])

print("Training Linear SVM...")

svm_pipeline.fit(X_train, y_train)

svm_preds = svm_pipeline.predict(X_test)

svm_acc = accuracy_score(y_test, svm_preds)
svm_macro_f1 = f1_score(y_test, svm_preds, average="macro")
svm_weighted_f1 = f1_score(y_test, svm_preds, average="weighted")

print("Linear SVM Results")
print("Accuracy:", svm_acc)
print("Macro F1:", svm_macro_f1)
print("Weighted F1:", svm_weighted_f1)

print("\nClassification Report:")
print(classification_report(y_test, svm_preds))

Training Linear SVM...
